# Import Packages

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType
from pyspark.sql.functions import input_file_name, col, lower, row_number, expr
from pyspark.sql.window import Window
from functools import reduce
from datetime import datetime

from apps.databricks.real_estate.notebooks.utils import pipeline_helpers


# Run Utils Functions

In [0]:
%run ../utils/pyutils

In [ ]:
%run ../utils/udf_helpers
# This will register SQL UDFs (e.g., `normalize_source_filename`) for use in DataFrame expressions and spark.sql() calls.

### Important: UDF Registration

- **Session-scoped:** UDFs registered via `spark.udf.register()` are available only for this session. They must be re-registered if the notebook is executed on a new cluster.
- **Delegation pattern:** All UDF implementations delegate to helper functions in `apps.databricks.real_estate.notebooks.utils.pipeline_helpers` to maintain a single source of truth for transformation logic.
- **Testing:** The underlying Python functions are tested locally with pytest (see `apps/databricks/real_estate/tests/test_pipeline_helpers.py`); UDF registration is validated at runtime in Databricks.

## Configuration


In [0]:
dbutils.widgets.text("raw_container", "raw", "Raw container name")
dbutils.widgets.text("silver_container", "silver", "Silver container name")
raw_container = dbutils.widgets.get("raw_container")
silver_container = dbutils.widgets.get("silver_container")
ds_raw = get_wasbs_path(container=raw_container)
ds_silver = get_wasbs_path(container=silver_container)
run_timestamp = datetime.utcnow().isoformat()
print(f"Run timestamp: {run_timestamp}")


## TODOs
- #TODO: Extract schema/path helpers into a Python module so we can write pytest-style unit tests (see first test goal).
- #TODO: Add simple pytest fixtures that mock `dbutils.widgets` and `input_file_name` to cover the widget/schema logic once helpers are extracted.


# Load Raw Data

## Generate Datasource Path

In [0]:
files = [f.path for f in dbutils.fs.ls(f"{ds_raw}/sold/all/") if f.path.endswith(".csv")]
print(f"Found {len(files)} raw sold files under {ds_raw}/sold/all/")


## Define Rename Column Settings

In [0]:
rename_dict = pipeline_helpers.RENAME_DICT
print(f"Loaded {len(rename_dict)} rename mappings")


## Define Input Schema


In [0]:
TYPE_MAP = {
    'IntegerType': IntegerType,
    'StringType': StringType,
    'DoubleType': DoubleType,
    'FloatType': FloatType,
}
sold_schema = StructType([
    StructField(name, TYPE_MAP[type_name](), True)
    for name, type_name in pipeline_helpers.SOLD_SCHEMA_FIELDS
])


## Load

In [0]:
dfs = [
    spark.read
         .format("csv")
         .option("header", True)
         .option("inferSchema", False)
         .schema(sold_schema)
         .load(path)
         .withColumn("source_file", input_file_name())
    for path in files
]


In [0]:
df_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    dfs
)

## Validate Source Data


In [0]:
if not files:
    raise ValueError(f"No raw CSV files found under {ds_raw}/sold/all/")
record_count = df_all.count()
missing_required = df_all.filter("booliId IS NULL OR soldDate IS NULL OR `soldPrice.raw` IS NULL").count()
print(f"[{run_timestamp}] Loaded {record_count} rows from {len(files)} files (missing required fields: {missing_required})")


In [0]:
df_all = df_all.select(*pipeline_helpers.SELECT_COLUMNS)


## Rename Columns

In [0]:
df_all = pipeline_helpers.rename_columns(df_all)
print("Columns renamed via helper map")


## Create Raw View

In [0]:
# Use the SQL-registered UDF `normalize_source_filename` (registered by `%run ../utils/udf_helpers`)
df_all = (
    df_all
    .withColumn("sourceFileName", expr("normalize_source_filename(source_file)"))
    .drop("source_file")
)


# Clean Data

## Subsetting

In [0]:
cast_map = pipeline_helpers.CAST_COLUMN_TYPES
for column, dtype in cast_map.items():
    df_all = df_all.withColumn(column, col(column).cast(dtype))

df_all = df_all.filter(~lower(col("url")).like("%annons%"))
for required in pipeline_helpers.REQUIRED_NON_NULL_COLUMNS:
    df_all = df_all.filter(col(required).isNotNull())

window_spec = Window.partitionBy("booliId", "soldDate").orderBy(col("soldPrice").desc())
df_clean = (
    df_all
    .withColumn("row_rank", row_number().over(window_spec))
    .filter(col("row_rank") == 1)
    .drop("row_rank")
)
df_clean.createOrReplaceTempView("sold_raw_subset_deduplicated")


## Deduplicating Raw Data

In [0]:
print("Deduplicated rows ready for merge")
df_clean.select("booliId", "soldDate", "soldPrice").limit(5).show()


# Create Delta Parquet Files & DB

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
query = f"""
CREATE TABLE IF NOT EXISTS silver.Fact_SoldObjects
(
  booliId INT,
  constructionYear INT,
  daysActive INT,
  soldDate TIMESTAMP,
  latitude FLOAT,
  longitude FLOAT,
  url STRING,
  typeName STRING,
  rent INT,
  floor FLOAT, 
  soldSqmPrice FLOAT,
  livingArea FLOAT,
  rooms FLOAT,
  listPrice INT,
  soldPrice INT,
  sourceFileName STRING
)
USING DELTA
LOCATION '{ds_silver}/soldObjects'
"""
spark.sql(query)

# Merge Data

In [0]:
%sql
MERGE INTO 
  silver.Fact_SoldObjects AS T
USING 
  sold_raw_subset_deduplicated AS S ON 
    T.booliId = S.booliId
    and T.soldDate = S.soldDate
WHEN MATCHED THEN UPDATE SET 
  *
WHEN NOT MATCHED THEN INSERT 
  *

# Logs

In [0]:
%sql
DESCRIBE HISTORY silver.Fact_SoldObjects

In [0]:
%sql
select * from silver.Fact_SoldObjects order by soldDate desc;

# Clean Up Raw Container From Raw Files

In [0]:
# Get the path to the "raw" container
raw_container_path = ds_raw

if len(files) > 0:
    dbutils.fs.rm(f"{raw_container_path}/sold/", recurse=True)
else:
    print("Skipping cleanup; no raw files processed.")
